In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
print("=" * 60)
print("DATA CLEANING PIPELINE")
print("=" * 60)

# STEP 1: Load the raw data
print("\n[1/5] Loading raw data...")
raw_path = r'/Users/siddharthm/Desktop/churn-prediction/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(raw_path)
print(f"✓ Loaded {len(df)} customers with {len(df.columns)} columns")

# STEP 2: Fix TotalCharges column (it's stored as string with blanks)
print("\n[2/5] Fixing TotalCharges column...")
print(f"   Current type: {df['TotalCharges'].dtype}")

# Count blank values
blank_count = (df['TotalCharges'] == ' ').sum()
print(f"   Found {blank_count} blank values")

# Convert to numeric (blanks become NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Impute missing values: use tenure * MonthlyCharges
# FIXED: Don't use inplace=True - assign the result back instead
df['TotalCharges'] = df['TotalCharges'].fillna(df['tenure'] * df['MonthlyCharges'])
print(f"✓ Fixed TotalCharges - imputed {blank_count} missing values")

# ADDED: Verify imputation worked
remaining_nans = df['TotalCharges'].isnull().sum()
print(f"   Remaining NaNs in TotalCharges: {remaining_nans}")
if remaining_nans > 0:
    print(f"   ⚠️ WARNING: Still have {remaining_nans} missing values!")

In [ ]:
# STEP 3: Consolidate redundant category values
# NEW STEP - This improves data quality for modeling
print("\n[3/5] Consolidating category values...")

# Columns like OnlineSecurity, TechSupport, etc. have 'No internet service'
# instead of just 'No'. Same for MultipleLines with 'No phone service'.
# Consolidate these for cleaner modeling.

internet_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                 'TechSupport', 'StreamingTV', 'StreamingMovies']

replaced_count = 0
for col in internet_cols:
    count = (df[col] == 'No internet service').sum()
    df[col] = df[col].replace('No internet service', 'No')
    replaced_count += count
print(f"   Replaced {replaced_count} 'No internet service' → 'No' across {len(internet_cols)} columns")

phone_count = (df['MultipleLines'] == 'No phone service').sum()
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')
print(f"   Replaced {phone_count} 'No phone service' → 'No' in MultipleLines")

print(f"✓ Consolidated {replaced_count + phone_count} redundant category values")

In [ ]:
# STEP 4: Add helpful columns
print("\n[4/5] Adding metadata columns...")

# Tenure groups (for easier analysis later)
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 73],
    labels=['0-12mo', '12-24mo', '24-48mo', '48+mo'],
    include_lowest=True
)

# Monthly charges quartile
df['charges_quartile'] = pd.qcut(
    df['MonthlyCharges'],
    q=4,
    labels=['Q1_Low', 'Q2_Medium', 'Q3_High', 'Q4_VeryHigh']
)

# Churn as binary (0/1) - easier for modeling later
df['churn_binary'] = (df['Churn'] == 'Yes').astype(int)

print("✓ Added 3 new columns: tenure_group, charges_quartile, churn_binary")

In [ ]:
# STEP 5: Save cleaned data
print("\n[5/5] Saving cleaned data...")
output_path = r'/Users/siddharthm/Desktop/churn-prediction/data/processed/telco_cleaned.csv'
Path(output_path).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f"✓ Saved to: {output_path}")

In [ ]:
# Print summary
print("\n" + "=" * 60)
print("CLEANING COMPLETE!")
print("=" * 60)
print(f"\nOriginal shape: (7043, 21)")
print(f"Cleaned shape:  {df.shape}")
print(f"\nChurn rate: {(df['Churn']=='Yes').sum()/len(df)*100:.2f}%")
print(f"Missing values: {df.isnull().sum().sum()}")
print("\nSample of cleaned data:")
print(df[['customerID', 'tenure', 'tenure_group', 'MonthlyCharges', 
          'TotalCharges', 'Churn', 'churn_binary']].head())

print("\n✓ You can now proceed to generating synthetic data!")

Verification Check

In [ ]:
# Check for any remaining missing values
print("Missing Values by Column:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "✓ No missing values!")

In [ ]:
# Check that category consolidation worked
print("Unique values in internet service columns (should only have Yes/No):")
print("\nOnlineSecurity:", df['OnlineSecurity'].unique())
print("TechSupport:", df['TechSupport'].unique())
print("MultipleLines:", df['MultipleLines'].unique())

In [ ]:
# Verify TotalCharges is numeric
print("Data types check:")
print(f"TotalCharges dtype: {df['TotalCharges'].dtype}")
print(f"MonthlyCharges dtype: {df['MonthlyCharges'].dtype}")
print(f"tenure dtype: {df['tenure'].dtype}")

if df['TotalCharges'].dtype == 'float64':
    print("\n✓ TotalCharges successfully converted to numeric!")
else:
    print("\n⚠️ WARNING: TotalCharges is not numeric!")